# M21 — error-accumulation control and minimal Gram load (train-only, T4)

Two controls on CIFAR-100 **validation** data:

1. **Minimal load.** On the Priority-3 FLY-CL stream (width 10,000), the direct INT8 Gram
   matrix gets the smallest diagonal load on a fixed grid for which FP32 Cholesky succeeds
   (no certificate). Exact, SRQ-INT8 and the Weyl repair are rerun and must reproduce Priority 3.
2. **Accumulation.** With the RanPAC head (widths 20,000 on three streams and 10,000 on one),
   recursive SRQ-INT8 is compared with a one-shot counterfactual that quantizes the exact factor
   of the current system once with the same codec.

**No `test.pt` is ever created or read.** Thresholds and paper consequences are frozen in
`docs/research/SRQ_GENERALIZATION_M21_PROTOCOL.md`.

Select a **T4 GPU**, upload `srq_generalization_m6_width_sweep_train_only.zip` when asked,
then run all cells top to bottom. The long cell runs one atomic unit per subprocess and can
be re-run in the same runtime to resume.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='beb94cb0ba5f3cf98f93dd8594575a87eb4f0bd3'
WORK_DIR='/content/SOHO-CL'
RUN_ROOT='/content/srq_m21'
FEATURE_CACHE_DIR=RUN_ROOT+'/cifar_train_features'
WTA_CACHE_DIR=RUN_ROOT+'/wta_10000'
OUTPUT_DIR=RUN_ROOT+'/output'
CONFIG='configs/srq_generalization_m21_accumulation_gram_load_train_only.json'
RUNNER='tools/srq_generalization_m21.py'
PROTOCOL='docs/research/SRQ_GENERALIZATION_M21_PROTOCOL.md'
M6_NAME='srq_generalization_m6_width_sweep_train_only.zip'
M6_SHA='b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e'
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
TRAIN_CACHE_SHA='ba53b82123964708123fe868a69d688bfdda4d66b1a3b7ea56329cc9ee9321dc'
TOTAL_UNITS=8
BATCH_SIZE=128
NUM_WORKERS=2
FINAL_EXPORT='/content/srq_generalization_m21_accumulation_gram_load_t4.zip'
assert REPO_COMMIT!='REPLACE_WITH_M21_COMMIT','Pin the commit that contains the M21 files.'

In [ ]:
# Clean pinned checkout, T4 check, and newline-normalized source lock.
import hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
def sha_raw(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
def run_visible(command):
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1','PYTHONDONTWRITEBYTECODE':'1'})
    assert process.stdout is not None
    for line in process.stdout: print(line,end='')
    returncode=process.wait()
    if returncode: raise RuntimeError(f'Command failed ({returncode}): {command}')
os.chdir('/content')
if Path(WORK_DIR).exists(): shutil.rmtree(WORK_DIR)
subprocess.run(['git','clone','--no-checkout','--quiet',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach','--quiet',REPO_COMMIT],cwd=WORK_DIR,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK_DIR,text=True).strip()==REPO_COMMIT
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Enable a Colab GPU and restart from cell 1.'
gpu_name=torch.cuda.get_device_name(0); gpu_bytes=torch.cuda.get_device_properties(0).total_memory
assert 'T4' in gpu_name,f'M21 requires Tesla T4 to stay comparable with the paper; found {gpu_name!r}.'
EXPECTED_SOURCE={
 'configs/srq_generalization_m21_accumulation_gram_load_train_only.json':'61bb9bbffca0ae2541ddf44d8c3056b78dfab54cc5fc5a54b2ea6e2fba2bbf99',
 'tools/srq_generalization_m21.py':'6f242b71ddf71405b88516a7ab04874ba9ac5ecf2f7294223602420b1eace9ed',
 'methods/srq_fly_optimized/minimal_load_control.py':'98e2fa72c8a66395df16cde2b4630034f50cbb6ca42e291fb1346e58788b87b1',
 'docs/research/SRQ_GENERALIZATION_M21_PROTOCOL.md':'33208b505184e0d034392a1306ca8197b823f6cb4ca72ec9bb3070a63f086876',
 'tests/test_srq_generalization_m21.py':'9cdd47a17dccaa5289210b9676a63696c914fde40d6f3909967fbe409cd43666',
 'tests/test_minimal_load_control.py':'a30eb9dec5f4ee18c1147509a3521f11b7c930c0b60c10ba8f7c3bace7eb76a1',
 'configs/srq_fly_priority3_direct_control_cifar100_train_only.json':'909fbd4da753018c3e8b3e8dcd8ea33e78dc5c5f6f32880035875d2b2b51e4dd',
 'tools/srq_fly_priority3_direct_control.py':'ca68754d50fa0e5de467064d519230adb12f5fb28440d85144a399b733ef3b80',
 'methods/srq_fly_optimized/direct_control.py':'84a680eadaac5dbd80091a47dfb66cb0dadddc9524356b6ea20b36405e58dbf0',
 'methods/srq_fly_optimized/learner.py':'40edac2e2cc88faac549f5c87217f3143d815bf53ecad8a37dfdb22c112691ae',
 'methods/srq_fly_optimized/storage.py':'9d288a3661985da657371e8581f406825d4a8d5e6e0c63381aacda8484490986',
 'tools/srq_fly_priority1_ablation.py':'b65ce01bfc2e2f9f07a61ecd056637156b504626c74c45b6aa3a7c8162e22136',
 'tools/srq_fly_d0.py':'60566c92512a97ae49d981b746f6c91c477b3d8dade3194ec6fa4b51f9c3619a',
 'tools/tail_fly_phasea.py':'efca2621e640ce329513e1a642d97042b974375496edcdf35169e9a594bfea73',
 'tools/twa_fly_pilot.py':'ee1efe6f793ea7ec6ba5ba4489dafefa76a8c2342447a3076bb3fae0316f088a',
 'methods/analytic_ridge/backends.py':'cb97a6b65991e41af5f52302bcbdeac5ded6dc95774055c64c6eecdbbfb50ad3',
 'methods/analytic_ridge/qr.py':'19d24d887e60ee7fc7adb1a2265253ea261c90636aef241241e5c6faad3a507b',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m6.py':'bad119dca8b2c6e78200c81917c8e8b03a5c50923135f951d8723fd5afd2ae61',
 'tools/srq_generalization_m11.py':'b13f0ad5ed81c33a61ecca53ff536bed3f7dd8c8b0731ef4cfe5a1a8adfe0651',
 'tools/srq_generalization_m20.py':'da2d86aa62733276dbb02db20a39b5fc3a7d056971eeb8e7f2836ae551363206',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'models/flyhash.py':'24ba321a71f735031b0da430ab4d3519e54e6c3149fc6c913c63b4172f6712cb',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for relative,expected in EXPECTED_SOURCE.items(): assert sha_source(relative)==expected,(relative,sha_source(relative),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
print('M21 PINNED SOURCE: PASS | GPU:',gpu_name,f'{gpu_bytes/2**30:.2f} GiB')

In [ ]:
# CPU preflight: minimal-load search and learner, tiny end-to-end units of both parts
# (the recursive arm must reproduce standalone backends), and the locked Priority-3 and
# backend regression tests, before any GPU time is spent.
run_visible([sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_minimal_load_control.py','tests/test_srq_generalization_m21.py','tests/test_srq_fly_priority3_direct_control.py','tests/test_analytic_ridge_backend.py'])
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('M21 PREFLIGHT TESTS: PASS')

In [ ]:
# Upload the immutable M6 artifact; stream s2025 is identity-locked to it.
from google.colab import files
os.chdir('/content')
uploaded=files.upload()
assert set(uploaded)=={M6_NAME},f'Upload exactly {M6_NAME}; got {sorted(uploaded)}'
M6_ARTIFACT=str((Path('/content')/M6_NAME).resolve())
assert sha_raw(M6_ARTIFACT)==M6_SHA,(sha_raw(M6_ARTIFACT),M6_SHA)
os.chdir(WORK_DIR)
print('M6 ARTIFACT VERIFIED')

In [ ]:
# Locked backbone checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE
assert sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH,'| CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only; held-out features are forbidden.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    run_visible([sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',RUN_ROOT+'/unused','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)])
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
assert sha_raw(cache/'train.pt')==TRAIN_CACHE_SHA
print('TRAIN CACHE READY | test.pt absent')

## Long cell: 8 atomic units

Order: the four FLY-CL minimal-load units (Exact, SRQ-INT8, Weyl repair, minimal load; the
first one also builds the WTA code cache), then the RanPAC accumulation units at width 20,000
(streams s2025, s2026, s2027) and width 10,000 (s2025). Each subprocess completes exactly one
unit and writes it atomically. If the runtime disconnects, re-run cells 2, 4, 5 and 6, then
this cell. Do not change thresholds, loads, streams or seeds after seeing any unit.

In [ ]:
unit_dir=Path(OUTPUT_DIR)/'units'; unit_dir.mkdir(parents=True,exist_ok=True)
def done(): return len(list(unit_dir.glob('*.json')))
while done() < TOTAL_UNITS:
    before=done()
    print(f'M21 START/RESUME: {before}/{TOTAL_UNITS} units complete',flush=True)
    run_visible([sys.executable,'-B',RUNNER,'--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--code-cache-dir',WTA_CACHE_DIR,'--source-m6-artifact',M6_ARTIFACT,'--output-dir',OUTPUT_DIR,'--device','cuda','--max-new-units','1','--require-clean-git'])
    after=done()
    assert after==before+1,f'Expected one new atomic unit; got {before} -> {after}'
assert Path(OUTPUT_DIR,'m21_results.json').is_file()
print(f'M21 ALL {TOTAL_UNITS} UNITS COMPLETE')

In [ ]:
# Report, then export every result even when a structural gate warns.
from google.colab import files
result_path=Path(OUTPUT_DIR)/'m21_results.json'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('GATES:',json.dumps(result['gates'],indent=2))
gram=result['gram_load']
print('GRAM-LOAD VERDICT:',gram['verdict'])
print('VALIDATION AIA:',json.dumps(gram['validation_aia_percent'],indent=2))
print('PRIORITY-3 REPRODUCTION |diff| pp:',gram['priority3_reproduction_abs_difference_pp'])
if gram['observations']:
    print('MINIMAL LOAD / LAMBDA:',[round(x,3) for x in gram['observations']['minimal_loading_to_ridge_ratio_by_task']])
    print('WEYL LOAD / LAMBDA:',[round(x,1) for x in gram['observations']['weyl_loading_to_ridge_ratio_by_task']])
acc=result['accumulation']
print('ACCUMULATION LOGIT VERDICT:',acc['logit_error_verdict'])
print('ACCUMULATION ACCURACY VERDICT:',acc['accuracy_verdict'])
print('PRIMARY WIDTH SUMMARY:',json.dumps(acc['primary'],indent=2))
for unit_id,row in acc['per_unit'].items():
    ratio=row['final_ratio_recursive_over_one_shot']
    print(f"{unit_id:18} logit ratio {ratio['relative_logit_error']:.3f} | system ratio {ratio['relative_system_action_error']:.3f} | AIA loss rec {row['aia_loss_pp']['recursive_int8']:+.3f} one {row['aia_loss_pp']['one_shot_int8']:+.3f}")
manifest={'schema_version':1,'study_id':result['study_id'],'repo_commit':REPO_COMMIT,'config_sha256':EXPECTED_SOURCE[CONFIG],'runner_sha256':EXPECTED_SOURCE[RUNNER],'minimal_load_module_sha256':EXPECTED_SOURCE['methods/srq_fly_optimized/minimal_load_control.py'],'result_sha256':sha_raw(result_path),'status':result['status']}
with zipfile.ZipFile(FINAL_EXPORT,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in sorted(Path(OUTPUT_DIR).rglob('*')):
        if path.is_file(): archive.write(path,'results/'+str(path.relative_to(OUTPUT_DIR)).replace('\\','/'))
    for relative in (CONFIG,PROTOCOL):
        archive.write(relative,'source/'+relative)
    archive.writestr('M21_ARTIFACT_MANIFEST.json',json.dumps(manifest,indent=2)+'\n')
print('FINAL EXPORT:',FINAL_EXPORT,'SHA256:',sha_raw(FINAL_EXPORT),'SIZE:',Path(FINAL_EXPORT).stat().st_size)
files.download(FINAL_EXPORT)
if result['status']!='PASS_M21_ACCUMULATION_GRAM_LOAD_TRAIN_ONLY': print('WARNING: preserve the artifact; do not relax gates or rerun a unit after seeing results.')